In [ ]:
def train_gan(generator, discriminator, dataloader, epochs=10, latent_dim=100):
    criterion = nn.BCELoss()
    lr = 0.0002
    beta1 = 0.5
    
    opt_g = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, 0.999))
    opt_d = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, 0.999))
    
    g_losses = []
    d_losses = []
    
    for epoch in range(epochs):
        for batch_idx, (real_images, _) in enumerate(dataloader):
            real_images = real_images.to(device)
            batch_size = real_images.size(0)
            
            # Labels for real and fake
            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)
            
            # Train Discriminator
            opt_d.zero_grad()
            
            # Real images
            real_output = discriminator(real_images)
            d_real_loss = criterion(real_output, real_labels)
            
            # Fake images
            z = torch.randn(batch_size, latent_dim, 1, 1).to(device)
            fake_images = generator(z)
            fake_output = discriminator(fake_images.detach())
            d_fake_loss = criterion(fake_output, fake_labels)
            
            d_loss = d_real_loss + d_fake_loss
            d_loss.backward()
            opt_d.step()
            
            # Train Generator
            opt_g.zero_grad()
            
            z = torch.randn(batch_size, latent_dim, 1, 1).to(device)
            fake_images = generator(z)
            fake_output = discriminator(fake_images)
            g_loss = criterion(fake_output, real_labels)
            
            g_loss.backward()
            opt_g.step()
            
            if batch_idx % 100 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] Step [{batch_idx}] - G Loss: {g_loss.item():.4f}, D Loss: {d_loss.item():.4f}")
        
        g_losses.append(g_loss.item())
        d_losses.append(d_loss.item())
    
    return g_losses, d_losses

print("GAN training function defined")

## 2. GAN Training Loop

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Generator Network
class Generator(nn.Module):
    def __init__(self, latent_dim=100, img_channels=3):
        super(Generator, self).__init__()
        self.latent_dim = latent_dim
        
        self.main = nn.Sequential(
            # Input: latent_dim x 1 x 1
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            # 512 x 4 x 4
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            # 256 x 8 x 8
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            # 128 x 16 x 16
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            # 64 x 32 x 32
            nn.ConvTranspose2d(64, img_channels, 4, 2, 1, bias=False),
            nn.Tanh()
            # img_channels x 64 x 64
        )
    
    def forward(self, z):
        return self.main(z)

# Discriminator Network
class Discriminator(nn.Module):
    def __init__(self, img_channels=3):
        super(Discriminator, self).__init__()
        
        self.main = nn.Sequential(
            # Input: img_channels x 64 x 64
            nn.Conv2d(img_channels, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # 64 x 32 x 32
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # 128 x 16 x 16
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            # 256 x 8 x 8
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            # 512 x 4 x 4
            nn.Conv2d(512, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
            # 1 x 1 x 1
        )
    
    def forward(self, x):
        return self.main(x)

# Initialize models
generator = Generator(latent_dim=100).to(device)
discriminator = Discriminator().to(device)

print("Generator and Discriminator created successfully")
print(f"Generator params: {sum(p.numel() for p in generator.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in discriminator.parameters()):,}")

## 1. DCGAN Architecture

# Generative Models and GANs
## Generating Synthetic Images and Image-to-Image Translation

Topics covered:
- Generative Adversarial Networks (GANs) fundamentals
- DCGAN (Deep Convolutional GAN)
- Conditional GANs (cGAN)
- StyleGAN and style transfer
- CycleGAN for unpaired translation
- Variational Autoencoders (VAE)
- Mode collapse and training stability
- Image super-resolution with GANs